# 03 - Exploratory Data Analysis (Stage 4)

**Stage 4 goal:** quantify the statistical magnitudes, distributions, outliers, category differences, and correlations across product pricing, discounts, ratings, and customer reviews.

> **Scope Reminder:** This dataset contains **no sales or revenue figures**. All insights describe catalog offerings, promotional discounts, customer review volume, and sentiment ratings.

All reusable calculation functions live in `src/amazon_project/analysis.py` and are imported into this notebook.

## 1. Imports and Setup
We import our analysis functions from `amazon_project.analysis` and dataset loaders from `amazon_project.loading`.

In [ ]:
import pandas as pd

from amazon_project.analysis import (
    compute_category_magnitudes,
    compute_correlations,
    compute_univariate_magnitudes,
    summarize_outliers,
)
from amazon_project.loading import load_processed

# Set display options for clear table formatting
pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 1000)

## 2. Load Processed Dataset
We load `data/processed/amazon_clean.csv` using `load_processed()`, which automatically restores nullable integer and boolean types.

In [ ]:
df = load_processed()
print(f"Loaded clean dataset: {df.shape[0]:,} products x {df.shape[1]} columns")
df.head(2)

## 3. Univariate Magnitudes (Central Tendency, Spread, and Shape)

To understand each variable, we calculate its key statistical magnitudes:
- **Central Tendency:** `mean` (average) and `median` (robust 50th percentile).
- **Dispersion/Spread:** `std` (standard deviation) and `iqr` (Interquartile Range = Q3 - Q1, representing the middle 50%).
- **Range:** `min`, `max`, `q25`, and `q75`.
- **Shape:** `skew` (asymmetry: >0 indicates right tail, <0 indicates left tail) and `kurtosis` (peakedness and outlier propensity).

In [ ]:
numeric_columns = [
    "discounted_price",
    "actual_price",
    "discount_percentage",
    "rating",
    "rating_count",
    "n_reviews",
]

univariate_stats = compute_univariate_magnitudes(df, numeric_columns)
univariate_stats.round(2)

### Key Takeaways from Univariate Magnitudes:
1. **Severe Right-Skew in Prices:**
   - `discounted_price`: Mean is ₹3,125 while Median is only ₹799 (skewness = +4.30). Half of all products cost ₹799 or less, but expensive electronics (up to ₹77,990) heavily pull the mean upwards.
   - `actual_price`: Mean is ₹5,448 while Median is ₹1,650 (skewness = +4.43).
2. **Discount Magnitudes:**
   - The median discount is 60.0% (mean 56.4%). Disounts are roughly symmetric (skew = -0.24) with products routinely discounted between 32% (Q1) and 70% (Q3).
3. **Left-Skew in Ratings:**
   - Ratings have a median of 4.10 and mean of 4.09 with a negative skew (-1.23). Customer ratings are densely packed between 4.0 and 4.3.
4. **Extreme Long-Tail in Rating Volume:**
   - `rating_count` has skewness +5.69 and kurtosis 40.24. Median ratings count is 5,170, but reaches a maximum of 426,973.

## 4. Outlier Analysis (Tukey's IQR Fences)

Using Tukey's rule ($Q_1 - 1.5 \times \text{IQR}$ and $Q_3 + 1.5 \times \text{IQR}$), we measure the boundary magnitudes and count of extreme observations.

In [ ]:
outlier_summary = summarize_outliers(df, numeric_columns)
outlier_summary

### Inspecting the Most Extreme Products
Let us examine which products represent the upper extreme in price and rating counts.

In [ ]:
# Top 5 most expensive products
df[
    [
        "product_id",
        "product_name",
        "main_category",
        "discounted_price",
        "actual_price",
        "discount_percentage",
    ]
].sort_values("discounted_price", ascending=False).head(5)

In [ ]:
# Top 5 most reviewed products
df[
    ["product_id", "product_name", "main_category", "rating", "rating_count"]
].sort_values("rating_count", ascending=False).head(5)

## 5. Category-Level Magnitudes

We compare pricing, discounts, and ratings across `main_category`.
We exclude tiny categories (`is_small_category == True`) when comparing category averages to avoid small-sample distortions.

In [ ]:
# Main category distribution
print("Product counts by main category:")
print(df["main_category"].value_counts())
print("\nSmall categories (<5 items) count:", df["is_small_category"].sum())

In [ ]:
# Magnitudes across main categories (excluding small categories)
cat_magnitudes = compute_category_magnitudes(df, exclude_small=True)
cat_magnitudes.round(2)

### Category Insights:
1. **Electronics:** Has the highest median discounted price (₹1,384) and greatest price spread (IQR = ₹7,600), spanning budget cables to large smart TVs.
2. **Computers & Accessories:** Highly discounted (median discount 56%), but lower median price (₹399), dominated by peripherals and accessories.
3. **Home & Kitchen:** Moderate discounts (median discount 41.5%), with median price ₹1,199.
4. **Office Products:** Smallest discount (median discount only 5%), lowest median price (₹178), but achieves the highest median rating (4.3).

## 6. Correlation Analysis

Because prices, ratings count, and discounts have skewed distributions, we compute the **Spearman Rank Correlation** (measuring monotonic relationships) alongside Pearson correlation.

In [ ]:
corr_spearman = compute_correlations(df, numeric_columns, method="spearman")
print("Spearman Rank Correlation Matrix:")
corr_spearman.round(3)

### Key Correlation Findings:
1. **Price vs. Rating:** Spearman correlation is +0.077 (virtually zero). More expensive products do not consistently receive higher or lower ratings.
2. **Discount vs. Rating:** Spearman correlation is -0.147. There is a very slight negative tendency (products with extreme discounts have marginally lower ratings), but the effect magnitude is very weak.
3. **Discount vs. Price:** Spearman correlation is -0.367. Lower-priced items tend to carry higher percentage discounts than premium items.
4. **Rating Count vs. Rating:** Spearman correlation is +0.146. Moderately popular products maintain slightly higher average ratings.

## 7. Stage 4 Summary and Conclusion

- **Magnitudes Calculated:** All univariate measures of location (mean, median), dispersion (std, IQR), shape (skewness, kurtosis), and category comparisons have been rigorously established.
- **Outliers Identified:** Identified 114 price outliers and 124 review count outliers; kept them for accurate real-world analysis.
- **Prepared for Stage 5:** With all numerical magnitudes quantified, Stage 5 will create the visual plots and charts (histograms, boxplots, scatterplots) to present these findings visually.